# LMP Controlled Evaluation Notebook

This notebook accompanies the paper on **Language-Mass Precision (LMP)**.

LMP is a **quotient-based model-language precision measure**. It instantiates the monotone language-quotient view of precision with a fixed, full-support probability mass over traces:

\[
\mathrm{LMP}_\mu(L,M)=
\frac{\mu(\widetilde L\cap\mathcal L(M))}{\mu(\mathcal L(M))}.
\]

The quotient construction is inherited from prior language-quotient work; the new ingredient is the **language measure** \(\mu\). The measure is not an empirical trace frequency and is not a stochastic execution semantics of the model. It is a declared reference measure that assigns finite mass to finite and infinite model languages.

## What this notebook is for

The notebook contains **controlled witness tests**, not a benchmark claiming empirical superiority of one discovery algorithm over another. The tests use small models because their relevant language relations are known by construction: language inclusion, flower-model generality, model-language equivalence, and log-extension monotonicity.

The notebook provides:

1. reusable functions for loading CSV logs and PNML Petri nets;
2. bounded PM4Py playout for model-language generation;
3. one LMP implementation shared by all experiments;
4. PM4Py precision and fitness wrappers;
5. Python-native stochastic similarities based on Jensen--Shannon and Earth-Mover distances;
6. diagnostics for selected Tax-style precision axioms;
7. trace-level attribution of imprecision.

## What LMP reports

For fitting logs, LMP is the fraction of model-language mass supported by observed variants. Since \(\mu\) is additive over traces, imprecision decomposes as:

\[
1-\mathrm{LMP}_\mu(L,M)=
\sum_{\sigma\in\mathcal L(M)\setminus\widetilde L}
\frac{\mu(\sigma)}{\mu(\mathcal L(M))}.
\]

Thus, each unobserved-but-allowed trace has a concrete contribution to total imprecision, relative to the declared prior.


## 1. Install dependencies

Run this cell only if dependencies are missing.

In [1]:
# Uncomment if needed:
# %pip install pm4py pandas scipy rapidfuzz

# Optional, for faster optimal transport if desired:
# %pip install pot

# Optional process-mining stochastic conformance library:
# %pip install ebi-pm

## 2. Imports and capability check

In [2]:
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple
from collections import Counter
import math
import time
import warnings

import pandas as pd

try:
    import pm4py
    from pm4py.algo.simulation.playout.petri_net.algorithm import Variants
    from pm4py.algo.simulation.playout.petri_net.algorithm import apply as petri_playout
    from pm4py.algo.simulation.playout.petri_net.variants import extensive
    PM4PY_AVAILABLE = True
except Exception as exc:
    PM4PY_AVAILABLE = False
    print("PM4Py not available:", exc)

try:
    from scipy.optimize import linprog
    SCIPY_AVAILABLE = True
except Exception as exc:
    SCIPY_AVAILABLE = False
    print("SciPy not available:", exc)

try:
    import ot
    POT_AVAILABLE = True
except Exception:
    POT_AVAILABLE = False

try:
    from rapidfuzz.distance import Levenshtein
    RAPIDFUZZ_AVAILABLE = True
except Exception:
    RAPIDFUZZ_AVAILABLE = False

try:
    import ebi
    EBI_AVAILABLE = True
except Exception:
    EBI_AVAILABLE = False

Trace = Tuple[str, ...]

capabilities = pd.DataFrame([
    {"library": "PM4Py", "available": PM4PY_AVAILABLE, "used_for": "logs, PNML, playout, classical precision/fitness"},
    {"library": "SciPy", "available": SCIPY_AVAILABLE, "used_for": "linear-programming fallback for Earth-Mover similarity"},
    {"library": "POT", "available": POT_AVAILABLE, "used_for": "optional optimal transport solver"},
    {"library": "RapidFuzz", "available": RAPIDFUZZ_AVAILABLE, "used_for": "fast trace edit distance"},
    {"library": "Ebi", "available": EBI_AVAILABLE, "used_for": "optional process-mining stochastic conformance"},
])
capabilities

,library,available,used_for
0,PM4Py,True,"logs, PNML, playout, classical precision/fitness"
1,SciPy,True,linear-programming fallback for Earth-Mover si...
2,POT,True,optional optimal transport solver
3,RapidFuzz,True,fast trace edit distance
4,Ebi,True,optional process-mining stochastic conformance


## 3. Paths and global settings

In [3]:
BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CASE_ID_COL = "case:concept:name"
ACTIVITY_COL = "concept:name"
TIMESTAMP_COL = "time:timestamp"

DEFAULT_MAX_TRACE_LENGTH = 10
DEFAULT_GEOMETRIC_LAMBDA = 0.7
DEFAULT_POWER_LAW_ALPHA = 2.0

# Earth-Mover can be expensive because it solves an optimal transport problem.
# This threshold avoids accidental large computations.
MAX_EMD_SUPPORT_SIZE = 80

## 4. Data loading utilities

In [4]:
def require_pm4py():
    if not PM4PY_AVAILABLE:
        raise ImportError("PM4Py is required for this notebook. Install it with: pip install pm4py")


def load_csv_log_for_pm4py(
    csv_path: Path,
    case_id_col: str = CASE_ID_COL,
    activity_col: str = ACTIVITY_COL,
    timestamp_col: Optional[str] = TIMESTAMP_COL,
    sep: str = ",",
):
    """Load a CSV log and return a formatted DataFrame plus a PM4Py EventLog."""
    require_pm4py()
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV log not found: {csv_path}")

    df = pd.read_csv(csv_path, sep=sep)
    missing = {case_id_col, activity_col}.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns {sorted(missing)} in {csv_path}")

    if timestamp_col is not None and timestamp_col in df.columns:
        df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
        df = df.sort_values([case_id_col, timestamp_col])
        formatted = pm4py.format_dataframe(df, case_id=case_id_col, activity_key=activity_col, timestamp_key=timestamp_col)
    else:
        df = df.sort_values([case_id_col], kind="stable")
        formatted = pm4py.format_dataframe(df, case_id=case_id_col, activity_key=activity_col)

    event_log = pm4py.convert_to_event_log(formatted)
    return formatted, event_log


def extract_variant_counts(
    df: pd.DataFrame,
    case_id_col: str = CASE_ID_COL,
    activity_col: str = ACTIVITY_COL,
    timestamp_col: Optional[str] = TIMESTAMP_COL,
) -> Counter:
    """Return a Counter mapping trace variants to their frequency in the log."""
    if timestamp_col is not None and timestamp_col in df.columns:
        df = df.sort_values([case_id_col, timestamp_col])
    else:
        df = df.sort_values([case_id_col], kind="stable")

    counts = Counter()
    for _, case_df in df.groupby(case_id_col, sort=False):
        variant = tuple(case_df[activity_col].astype(str).tolist())
        counts[variant] += 1
    return counts


def load_petri_net_from_pnml(pnml_path: Path):
    """Load a Petri net, initial marking, and final marking from PNML."""
    require_pm4py()
    pnml_path = Path(pnml_path)
    if not pnml_path.exists():
        raise FileNotFoundError(f"PNML file not found: {pnml_path}")
    return pm4py.read_pnml(str(pnml_path))


def compute_bounded_language_with_playout(
    net,
    initial_marking,
    final_marking,
    max_trace_length: int = DEFAULT_MAX_TRACE_LENGTH,
    activity_key: str = ACTIVITY_COL,
) -> Set[Trace]:
    """Generate L_{<=K}(M) with PM4Py extensive playout."""
    require_pm4py()
    parameters = {
        extensive.Parameters.MAX_TRACE_LENGTH: max_trace_length,
        extensive.Parameters.ACTIVITY_KEY: activity_key,
    }
    generated_log = petri_playout(
        net,
        initial_marking,
        final_marking=final_marking,
        parameters=parameters,
        variant=Variants.EXTENSIVE,
    )
    language: Set[Trace] = set()
    for trace in generated_log:
        variant = tuple(event[activity_key] for event in trace)
        language.add(variant)
    return language

## 5. LMP implementation

This is the single Language-Mass Precision implementation used in all experiments. The power-law prior is normalized so that both geometric and power-law trace masses are probability masses over \(\Sigma^*\). Scores are unchanged by this normalization, but exported numerator and denominator masses now match the paper notation.


In [5]:
def infer_alphabet(*languages: Iterable[Trace]) -> Set[str]:
    alphabet: Set[str] = set()
    for language in languages:
        for trace in language:
            alphabet.update(trace)
    return alphabet


def zeta_normalizer(alpha: float) -> float:
    """Return zeta(alpha) for the normalized power-law length prior."""
    if alpha <= 1.0:
        raise ValueError("alpha must be > 1")

    try:
        from scipy.special import zeta as scipy_zeta
        return float(scipy_zeta(alpha, 1.0))
    except Exception:
        pass

    try:
        import mpmath as mp
        return float(mp.zeta(alpha))
    except Exception:
        pass

    if abs(alpha - 2.0) < 1e-12:
        return math.pi ** 2 / 6.0

    # Portable fallback. This is sufficient for notebook use, but production code
    # should use scipy.special.zeta or another reliable implementation.
    total = 0.0
    n = 1
    while n <= 2_000_000:
        term = n ** (-alpha)
        total += term
        if term < 1e-14:
            break
        n += 1
    return total


def trace_mass(
    trace: Trace,
    alphabet_size: int,
    prior: str = "power_law",
    lambda_: float = DEFAULT_GEOMETRIC_LAMBDA,
    alpha: float = DEFAULT_POWER_LAW_ALPHA,
) -> float:
    """Probability mass of one trace under the declared LMP prior."""
    if alphabet_size <= 0:
        raise ValueError("alphabet_size must be positive")
    n = len(trace)
    if prior == "geometric":
        if not (0.0 < lambda_ < 1.0):
            raise ValueError("lambda_ must be in (0, 1)")
        return (1.0 - lambda_) * (lambda_ ** n) / (alphabet_size ** n)
    if prior == "power_law":
        if alpha <= 1.0:
            raise ValueError("alpha must be > 1")
        return ((n + 1) ** (-alpha)) / (zeta_normalizer(alpha) * (alphabet_size ** n))
    raise ValueError(f"Unknown prior: {prior!r}")


def language_mass(
    traces: Iterable[Trace],
    alphabet_size: int,
    prior: str = "power_law",
    lambda_: float = DEFAULT_GEOMETRIC_LAMBDA,
    alpha: float = DEFAULT_POWER_LAW_ALPHA,
) -> float:
    return sum(trace_mass(t, alphabet_size, prior=prior, lambda_=lambda_, alpha=alpha) for t in traces)


def compute_lmp(
    log_variants: Set[Trace],
    model_language: Set[Trace],
    alphabet: Optional[Set[str]] = None,
    prior: str = "power_law",
    lambda_: float = DEFAULT_GEOMETRIC_LAMBDA,
    alpha: float = DEFAULT_POWER_LAW_ALPHA,
) -> Dict[str, object]:
    """Compute bounded LMP and useful diagnostics."""
    if alphabet is None:
        alphabet = infer_alphabet(log_variants, model_language)
    alphabet_size = len(alphabet)
    if alphabet_size == 0:
        raise ValueError("Empty alphabet")

    fitting = log_variants.intersection(model_language)
    missing_log = log_variants.difference(model_language)
    extra_model = model_language.difference(log_variants)

    numerator = language_mass(fitting, alphabet_size, prior=prior, lambda_=lambda_, alpha=alpha)
    denominator = language_mass(model_language, alphabet_size, prior=prior, lambda_=lambda_, alpha=alpha)
    score = numerator / denominator if denominator > 0 else float("nan")

    return {
        "score": score,
        "numerator_mass": numerator,
        "denominator_mass": denominator,
        "num_fitting_variants": len(fitting),
        "num_missing_log_variants": len(missing_log),
        "num_extra_model_variants": len(extra_model),
        "fitting_variants": fitting,
        "missing_log_variants": missing_log,
        "extra_model_variants": extra_model,
    }


def imprecision_attribution(
    log_variants: Set[Trace],
    model_language: Set[Trace],
    alphabet: Optional[Set[str]] = None,
    prior: str = "power_law",
    lambda_: float = DEFAULT_GEOMETRIC_LAMBDA,
    alpha: float = DEFAULT_POWER_LAW_ALPHA,
) -> pd.DataFrame:
    """Return trace-level contributions to 1 - LMP.

    The function works on an explicit finite or bounded model language. For a
    genuinely infinite language, the same idea applies but the language must be
    represented symbolically or truncated with a certified tail bound.
    """
    if alphabet is None:
        alphabet = infer_alphabet(log_variants, model_language)
    alphabet_size = len(alphabet)
    denominator = language_mass(model_language, alphabet_size, prior=prior, lambda_=lambda_, alpha=alpha)
    rows = []
    for trace in sorted(model_language.difference(log_variants), key=lambda t: (len(t), t)):
        mass = trace_mass(trace, alphabet_size, prior=prior, lambda_=lambda_, alpha=alpha)
        rows.append({
            "trace": " -> ".join(trace),
            "length": len(trace),
            "trace_mass": mass,
            "imprecision_share": mass / denominator if denominator > 0 else float("nan"),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("imprecision_share", ascending=False).reset_index(drop=True)
        df["cumulative_imprecision_share"] = df["imprecision_share"].cumsum()
    return df


def median_trace_length(log_variants: Iterable[Trace]) -> float:
    lengths = sorted(len(trace) for trace in log_variants)
    if not lengths:
        raise ValueError("Cannot compute a median length for an empty log support")
    mid = len(lengths) // 2
    if len(lengths) % 2:
        return float(lengths[mid])
    return 0.5 * (lengths[mid - 1] + lengths[mid])


def lambda_from_median_length(ell_50: float) -> float:
    """Geometric prior parameter with half the length mass at <= ell_50."""
    if ell_50 < 0:
        raise ValueError("ell_50 must be non-negative")
    return 2 ** (-1.0 / (ell_50 + 1.0))


def power_law_length_cdf(k: int, alpha: float) -> float:
    if k < 0:
        return 0.0
    numerator = sum((n + 1) ** (-alpha) for n in range(k + 1))
    return numerator / zeta_normalizer(alpha)


def alpha_from_median_length(ell_50: float, lo: float = 1.000001, hi: float = 100.0) -> float:
    """Power-law alpha with half the length mass at <= ell_50.

    The solver uses a robust bisection search to avoid requiring SciPy.
    """
    k = int(round(ell_50))
    if k < 0:
        raise ValueError("ell_50 must be non-negative")

    def f(a: float) -> float:
        return power_law_length_cdf(k, a) - 0.5

    if f(lo) >= 0:
        return lo
    if f(hi) <= 0:
        return hi
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if f(mid) < 0:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


### Trace-level attribution example

The decomposition is easiest to read on a tiny explicit language. Suppose a model admits observed traces of total mass 0.50 and three unobserved-but-allowed traces with masses 0.10, 0.08, and 0.02. The model-language mass is 0.70, so:

\[
\mathrm{LMP}=0.50/0.70=0.714.
\]

The remaining imprecision is 0.286, and the three unseen traces contribute:

\[
0.10/0.70,\quad 0.08/0.70,\quad 0.02/0.70.
\]

This is what the paper means by **additive trace-level attribution**. A trace matters more when it receives more mass under the declared prior; this is a statement about the measurement scale, not a ground-truth probability of the process.

In [6]:
toy_denominator = 0.30 + 0.20 + 0.10 + 0.08 + 0.02
toy_lmp = (0.30 + 0.20) / toy_denominator
toy_contributions = pd.DataFrame([
    {"unseen_trace": "sigma_3", "trace_mass": 0.10, "imprecision_share": 0.10 / toy_denominator},
    {"unseen_trace": "sigma_4", "trace_mass": 0.08, "imprecision_share": 0.08 / toy_denominator},
    {"unseen_trace": "sigma_5", "trace_mass": 0.02, "imprecision_share": 0.02 / toy_denominator},
])
toy_contributions["cumulative_imprecision_share"] = toy_contributions["imprecision_share"].cumsum()
print("Toy LMP:", round(toy_lmp, 3))
display(toy_contributions)

Toy LMP: 0.714


,unseen_trace,trace_mass,imprecision_share,cumulative_imprecision_share
0,sigma_3,0.10,0.142857,0.142857
1,sigma_4,0.08,0.114286,0.257143
2,sigma_5,0.02,0.028571,0.285714


### Prior calibration helper

A practical default is to calibrate the length prior from the median observed trace length \(\ell_{50}\). For the geometric prior:

\[
\lambda_{50}=2^{-1/(\ell_{50}+1)}.
\]

The helper functions above implement this formula and the analogous numerical calibration for the power-law prior. The calibrated prior should be fixed before comparing models and then varied in a sensitivity sweep.

## 5a. Exact LMP for full flower languages

The core LMP definition is not bounded. The bounded PM4Py playout computes an approximation:

\[
LMP^{\leq K}(L,M)
=
\frac{\mu(\widetilde L\cap \mathcal L_{\leq K}(M))}
{\mu(\mathcal L_{\leq K}(M))}
\]

For finite languages this can coincide with the exact value. For a flower model, however,

\[
\mathcal L(M)=\Sigma^*
\]

and the denominator is known analytically. Therefore, the exact flower score should not be computed from the truncated playout denominator. The bounded value is an upper bound whenever the log variants are already included in the bounded language, because the truncated denominator omits unobserved tail mass.

In [7]:
def exact_full_sigma_star_mass(
    alphabet_size: int,
    prior: str = "power_law",
    lambda_: float = DEFAULT_GEOMETRIC_LAMBDA,
    alpha: float = DEFAULT_POWER_LAW_ALPHA,
) -> float:
    """Exact mass of Sigma* under the normalized LMP trace priors."""
    if alphabet_size <= 0:
        raise ValueError("alphabet_size must be positive")
    if prior == "geometric" and not (0.0 < lambda_ < 1.0):
        raise ValueError("lambda_ must be in (0, 1)")
    if prior == "power_law" and alpha <= 1.0:
        raise ValueError("alpha must be > 1")
    return 1.0


def compute_lmp_against_full_sigma_star(
    log_variants: Set[Trace],
    alphabet: Set[str],
    prior: str = "power_law",
    lambda_: float = DEFAULT_GEOMETRIC_LAMBDA,
    alpha: float = DEFAULT_POWER_LAW_ALPHA,
) -> Dict[str, object]:
    """Exact LMP for a full flower language Sigma*."""
    alphabet_size = len(alphabet)
    numerator = language_mass(
        log_variants,
        alphabet_size,
        prior=prior,
        lambda_=lambda_,
        alpha=alpha,
    )
    denominator = exact_full_sigma_star_mass(
        alphabet_size,
        prior=prior,
        lambda_=lambda_,
        alpha=alpha,
    )
    return {
        "score": numerator / denominator if denominator > 0 else float("nan"),
        "numerator_mass": numerator,
        "denominator_mass": denominator,
    }


## 6. Classical PM4Py precision and fitness wrappers

In [8]:
def timed_call(func, *args, **kwargs):
    start = time.perf_counter()
    try:
        value = func(*args, **kwargs)
        status = "ok"
    except Exception as exc:
        value = float("nan")
        status = f"ERROR: {type(exc).__name__}: {exc}"
    elapsed = time.perf_counter() - start
    return value, elapsed, status


def pm4py_metric_wrappers(event_log, net, im, fm) -> List[Dict[str, object]]:
    """Compute PM4Py metrics with safe wrappers."""
    wrappers = []
    if not PM4PY_AVAILABLE:
        return wrappers

    candidates = [
        ("pm4py_precision_token_based_replay", getattr(pm4py, "precision_token_based_replay", None)),
        ("pm4py_precision_alignments", getattr(pm4py, "precision_alignments", None)),
        ("pm4py_precision_footprints", getattr(pm4py, "precision_footprints", None)),
        ("pm4py_fitness_token_based_replay", getattr(pm4py, "fitness_token_based_replay", None)),
        ("pm4py_fitness_alignments", getattr(pm4py, "fitness_alignments", None)),
    ]

    for name, fn in candidates:
        if fn is None:
            wrappers.append({"metric": name, "score": float("nan"), "time_s": 0.0, "status": "unavailable"})
            continue
        score, elapsed, status = timed_call(fn, event_log, net, im, fm)
        wrappers.append({"metric": name, "score": score, "time_s": elapsed, "status": status})
    return wrappers

## 7. Stochastic metric utilities

The stochastic metrics compare distributions over trace variants.

They are not pure language-precision metrics. They answer a different question: whether the model distribution resembles the empirical log distribution.

In [9]:
def normalize_distribution(weights: Dict[Trace, float]) -> Dict[Trace, float]:
    total = float(sum(weights.values()))
    if total <= 0:
        return {k: 0.0 for k in weights}
    return {k: v / total for k, v in weights.items()}


def log_distribution_from_counts(counts: Counter) -> Dict[Trace, float]:
    return normalize_distribution({trace: float(count) for trace, count in counts.items()})


def model_distribution_uniform(model_language: Set[Trace]) -> Dict[Trace, float]:
    if not model_language:
        return {}
    p = 1.0 / len(model_language)
    return {trace: p for trace in model_language}


def model_distribution_prior(
    model_language: Set[Trace],
    alphabet: Set[str],
    prior: str = "power_law",
    lambda_: float = DEFAULT_GEOMETRIC_LAMBDA,
    alpha: float = DEFAULT_POWER_LAW_ALPHA,
) -> Dict[Trace, float]:
    alphabet_size = len(alphabet)
    weights = {
        trace: trace_mass(trace, alphabet_size, prior=prior, lambda_=lambda_, alpha=alpha)
        for trace in model_language
    }
    return normalize_distribution(weights)


def js_similarity(p: Dict[Trace, float], q: Dict[Trace, float], base: float = 2.0) -> float:
    """Return 1 - Jensen-Shannon divergence. With base 2, it lies in [0,1]."""
    support = set(p).union(q)
    def kl(a, b):
        s = 0.0
        for x in support:
            ax = a.get(x, 0.0)
            bx = b.get(x, 0.0)
            if ax > 0 and bx > 0:
                s += ax * (math.log(ax / bx) / math.log(base))
        return s
    m = {x: 0.5 * p.get(x, 0.0) + 0.5 * q.get(x, 0.0) for x in support}
    jsd = 0.5 * kl(p, m) + 0.5 * kl(q, m)
    return 1.0 - jsd


def trace_edit_distance(a: Trace, b: Trace) -> int:
    """Edit distance between traces."""
    if RAPIDFUZZ_AVAILABLE:
        return Levenshtein.distance(list(a), list(b))
    # fallback dynamic programming
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]


def normalized_trace_distance(a: Trace, b: Trace) -> float:
    denom = max(len(a), len(b), 1)
    return trace_edit_distance(a, b) / denom


def earth_mover_similarity(
    p: Dict[Trace, float],
    q: Dict[Trace, float],
    max_support_size: int = MAX_EMD_SUPPORT_SIZE,
) -> Tuple[float, str]:
    """Return 1 - EMD using normalized edit distance as ground metric."""
    support_p = [x for x, v in p.items() if v > 0]
    support_q = [x for x, v in q.items() if v > 0]
    if not support_p or not support_q:
        return float("nan"), "empty distribution"
    if len(support_p) * len(support_q) > max_support_size * max_support_size:
        return float("nan"), f"skipped: support too large ({len(support_p)} x {len(support_q)})"

    a = [p[x] for x in support_p]
    b = [q[x] for x in support_q]
    C = [[normalized_trace_distance(x, y) for y in support_q] for x in support_p]

    if POT_AVAILABLE:
        import numpy as np
        cost = ot.emd2(np.array(a, dtype=float), np.array(b, dtype=float), np.array(C, dtype=float))
        return 1.0 - float(cost), "ok"

    if SCIPY_AVAILABLE:
        import numpy as np
        m, n = len(support_p), len(support_q)
        c = np.array(C, dtype=float).reshape(-1)
        A_eq = []
        b_eq = []
        # row sums
        for i in range(m):
            row = np.zeros(m * n)
            for j in range(n):
                row[i * n + j] = 1.0
            A_eq.append(row); b_eq.append(a[i])
        # column sums
        for j in range(n):
            col = np.zeros(m * n)
            for i in range(m):
                col[i * n + j] = 1.0
            A_eq.append(col); b_eq.append(b[j])
        res = linprog(c, A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=(0, None), method="highs")
        if not res.success:
            return float("nan"), f"linprog failed: {res.message}"
        return 1.0 - float(res.fun), "ok"

    return float("nan"), "requires POT or SciPy"


def stochastic_metric_rows(
    log_counts: Counter,
    model_language: Set[Trace],
    alphabet: Set[str],
    prior: str = "power_law",
    lambda_: float = DEFAULT_GEOMETRIC_LAMBDA,
    alpha: float = DEFAULT_POWER_LAW_ALPHA,
) -> List[Dict[str, object]]:
    """Compute JS and EMD similarities for uniform and prior-weighted model distributions."""
    rows = []
    p_log = log_distribution_from_counts(log_counts)
    model_distributions = {
        "uniform": model_distribution_uniform(model_language),
        f"{prior}_prior": model_distribution_prior(model_language, alphabet, prior=prior, lambda_=lambda_, alpha=alpha),
    }

    for dist_name, p_model in model_distributions.items():
        start = time.perf_counter()
        score = js_similarity(p_log, p_model)
        elapsed = time.perf_counter() - start
        rows.append({"metric": f"stochastic_js_{dist_name}", "score": score, "time_s": elapsed, "status": "ok"})

        start = time.perf_counter()
        score, status = earth_mover_similarity(p_log, p_model)
        elapsed = time.perf_counter() - start
        rows.append({"metric": f"stochastic_emd_{dist_name}", "score": score, "time_s": elapsed, "status": status})

    return rows

## 8. Optional Ebi note

Ebi is a process-mining stochastic conformance library. Its integration depends on the stochastic model format used in the local environment. This notebook therefore implements Python-native Jensen-Shannon and Earth-Mover similarities by default.

If Ebi is installed, the capability table above will show it. You can add Ebi-specific calls in this section once the stochastic model format is selected.

In [10]:
if EBI_AVAILABLE:
    print("Ebi is available. Add Ebi-specific stochastic conformance calls here if the model format is ready.")
else:
    print("Ebi is not installed. The notebook will use Python-native stochastic metrics.")

Ebi is available. Add Ebi-specific stochastic conformance calls here if the model format is ready.


## 9. Experiment registry

## Compatibility helpers for stochastic distributions

These helpers are used by the simulation-derived stochastic metrics. They are included to keep this notebook self-contained even if some optional libraries are not installed.

In [11]:
from collections import Counter
import math
import random

def empirical_log_distribution(
    df: pd.DataFrame,
    case_id_col: str = CASE_ID_COL,
    activity_col: str = ACTIVITY_COL,
    timestamp_col: Optional[str] = TIMESTAMP_COL,
) -> Dict[Trace, float]:
    # Compute empirical probabilities over trace variants.
    variants = []
    if timestamp_col is not None and timestamp_col in df.columns:
        df_sorted = df.sort_values([case_id_col, timestamp_col])
    else:
        df_sorted = df.sort_values([case_id_col], kind="stable")
    for _, case_df in df_sorted.groupby(case_id_col, sort=False):
        variants.append(tuple(case_df[activity_col].astype(str).tolist()))
    counts = Counter(variants)
    total = sum(counts.values())
    return {trace: count / total for trace, count in counts.items()}

def jensen_shannon_similarity_arrays(p, q) -> float:
    # Jensen-Shannon similarity, 1 - JSD with log base 2.
    total_p = sum(p)
    total_q = sum(q)
    if total_p <= 0 or total_q <= 0:
        return float("nan")
    p = [x / total_p for x in p]
    q = [x / total_q for x in q]
    m = [(x + y) / 2 for x, y in zip(p, q)]

    def kl_div(a, b):
        value = 0.0
        for ai, bi in zip(a, b):
            if ai > 0:
                value += ai * math.log(ai / bi, 2)
        return value

    jsd = 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)
    return 1.0 - jsd

def normalized_trace_edit_distance(a: Trace, b: Trace) -> float:
    # Normalized Levenshtein distance between two activity sequences.
    if not a and not b:
        return 0.0
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            substitution = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + substitution,
            )
    return dp[n][m] / max(n, m)

def earth_mover_similarity_from_arrays(p, q, cost_matrix) -> float:
    # Earth-Mover similarity, computed as 1 - optimal transport cost.
    try:
        from scipy.optimize import linprog
    except Exception:
        return float("nan")
    total_p = sum(p)
    total_q = sum(q)
    if total_p <= 0 or total_q <= 0:
        return float("nan")
    p = [x / total_p for x in p]
    q = [x / total_q for x in q]
    n = len(p)
    m = len(q)
    c = [cost_matrix[i][j] for i in range(n) for j in range(m)]
    A_eq = []
    b_eq = []
    for i in range(n):
        row = [0.0] * (n * m)
        for j in range(m):
            row[i * m + j] = 1.0
        A_eq.append(row)
        b_eq.append(p[i])
    for j in range(m):
        row = [0.0] * (n * m)
        for i in range(n):
            row[i * m + j] = 1.0
        A_eq.append(row)
        b_eq.append(q[j])
    bounds = [(0.0, None)] * (n * m)
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        return float("nan")
    return 1.0 - float(res.fun)

## Structural stochastic semantics by simulation

The stochastic metrics already included in the notebook can be computed from a distribution over the generated visible language. In that case, two language-equivalent models receive the same stochastic distribution whenever the same language-level weighting rule is used.

This section adds a second option: estimate the model distribution by **simulation of the Petri net structure**.

At each marking \(m\), the simulator selects uniformly among the enabled transitions:

\[
P(t \mid m)=\frac{1}{|Enabled(m)|}
\]

Visible transition labels are appended to the generated trace, while silent transitions are fired but not recorded. After many simulation runs, the empirical model distribution is:

\[
\widehat P_M(\sigma)
=
\frac{\#\text{simulations producing }\sigma}
{\#\text{completed simulations}}
\]

This is intentionally different from the language-derived distribution. Two models may have the same visible language but different internal branching structures, and therefore different simulation-derived probability distributions.

In [12]:
if PM4PY_AVAILABLE:
    from pm4py.objects.petri_net import semantics
else:
    semantics = None

def is_final_marking(marking, final_marking) -> bool:
    # Equality is sufficient for the explicit final markings used in this benchmark.
    return marking == final_marking

def simulate_one_trace(
    net,
    initial_marking,
    final_marking,
    max_steps: int = 100,
    rng: Optional[random.Random] = None,
) -> Tuple[Optional[Trace], str]:
    # Simulate one execution by choosing uniformly among enabled transitions.
    require_pm4py()
    if rng is None:
        rng = random.Random()

    marking = initial_marking
    visible_trace: List[str] = []

    for _ in range(max_steps):
        if is_final_marking(marking, final_marking):
            return tuple(visible_trace), "completed"

        enabled = list(semantics.enabled_transitions(net, marking))
        if not enabled:
            return None, "deadlock"

        transition = rng.choice(enabled)

        if transition.label is not None:
            visible_trace.append(str(transition.label))

        marking = semantics.execute(transition, net, marking)

    if is_final_marking(marking, final_marking):
        return tuple(visible_trace), "completed"

    return None, "timeout"

def simulate_model_distribution(
    net,
    initial_marking,
    final_marking,
    n_simulations: int = 10000,
    max_steps: int = 100,
    seed: int = 42,
) -> Tuple[Dict[Trace, float], Dict[str, object]]:
    # Estimate a probability distribution over visible traces by simulation.
    rng = random.Random(seed)
    counts = Counter()
    status_counts = Counter()

    for _ in range(n_simulations):
        trace, status = simulate_one_trace(
            net,
            initial_marking,
            final_marking,
            max_steps=max_steps,
            rng=rng,
        )
        status_counts[status] += 1
        if status == "completed" and trace is not None:
            counts[trace] += 1

    completed = status_counts["completed"]
    distribution = {trace: count / completed for trace, count in counts.items()} if completed > 0 else {}

    diagnostics = {
        "n_simulations": n_simulations,
        "completed_runs": status_counts["completed"],
        "deadlocked_runs": status_counts["deadlock"],
        "timeout_runs": status_counts["timeout"],
        "completion_rate": status_counts["completed"] / n_simulations,
        "deadlock_rate": status_counts["deadlock"] / n_simulations,
        "timeout_rate": status_counts["timeout"] / n_simulations,
        "num_simulated_variants": len(distribution),
    }

    return distribution, diagnostics

def js_similarity_from_distributions(
    log_distribution: Dict[Trace, float],
    model_distribution: Dict[Trace, float],
) -> float:
    # Jensen-Shannon similarity between two explicit trace distributions.
    support = sorted(set(log_distribution).union(model_distribution))
    if not support:
        return float("nan")
    p = [log_distribution.get(trace, 0.0) for trace in support]
    q = [model_distribution.get(trace, 0.0) for trace in support]
    return jensen_shannon_similarity_arrays(p, q)

def emd_similarity_from_distributions(
    log_distribution: Dict[Trace, float],
    model_distribution: Dict[Trace, float],
) -> float:
    # Earth-Mover similarity between two explicit trace distributions.
    support_log = sorted(log_distribution)
    support_model = sorted(model_distribution)
    if not support_log or not support_model:
        return float("nan")
    p = [log_distribution[trace] for trace in support_log]
    q = [model_distribution[trace] for trace in support_model]
    cost_matrix = [
        [normalized_trace_edit_distance(a, b) for b in support_model]
        for a in support_log
    ]
    return earth_mover_similarity_from_arrays(p, q, cost_matrix)

In [13]:
EXPERIMENTS = [
    {
        "name": "precision_vs_fitness_three_models",
        "type": "model_comparison",
        "axiom": "precision_vs_fitness",
        "description": "M1 is smaller than the log, M2 equals the log language, M3 is larger.",
        "folder": DATA_DIR / "precision_fitness_three_models",
        "log": "simple_log.csv",
        "models": {
            "M1_smaller_than_log": "model_1_smaller_than_log.pnml",
            "M2_exact_log_language": "model_2_exact_log_language.pnml",
            "M3_larger_than_log": "model_3_larger_than_log.pnml",
        },
        "max_trace_length": 6,
        "checks": [{"type": "order", "name": "A2_subcase_M2_ge_M3", "order": ["M2_exact_log_language", "M3_larger_than_log"]}],
    },
    {
        "name": "axiom2_clean_two_models",
        "type": "model_comparison",
        "axiom": "A2",
        "description": "Clean A2 case: L~=L(M1) subset L(M2).",
        "folder": DATA_DIR / "axiom2_clean_two_models",
        "log": "axiom2_log.csv",
        "models": {
            "M1_exact_log_language": "model_1_constrained_exact_log_language.pnml",
            "M2_larger_language": "model_2_larger_language_superset.pnml",
        },
        "max_trace_length": 10,
        "checks": [{"type": "order", "name": "A2_M1_ge_M2", "order": ["M1_exact_log_language", "M2_larger_language"]}],
    },
    {
        "name": "axiom2_decoy_stress",
        "type": "model_comparison",
        "axiom": "A2",
        "description": "A2 stress test with dead visible branches in the smaller model language model.",
        "folder": DATA_DIR / "axiom2_decoy_stress",
        "log": "decoy_axiom2_log.csv",
        "models": {
            "M1_exact_language_with_decoys": "model_1_exact_language_with_decoys.pnml",
            "M2_clean_larger_language": "model_2_clean_larger_language.pnml",
        },
        "max_trace_length": 8,
        "checks": [{"type": "order", "name": "A2_decoy_M1_ge_M2", "order": ["M1_exact_language_with_decoys", "M2_clean_larger_language"]}],
    },
    {
        "name": "axiom3_flower",
        "type": "model_comparison",
        "axiom": "A3",
        "description": "Constrained model versus flower model over the same alphabet.",
        "folder": DATA_DIR / "axiom3_flower",
        "log": "flower_log.csv",
        "models": {
            "M_constrained_abc": "model_constrained_abc.pnml",
            "M_flower_abcd": "model_flower_abcd.pnml",
        },
        "max_trace_length": 3,
        "checks": [{"type": "order", "name": "A3_constrained_gt_flower", "order": ["M_constrained_abc", "M_flower_abcd"]}],
    },
    {
        "name": "axiom4_equivalent",
        "type": "model_comparison",
        "axiom": "A4",
        "description": "Two structurally different models with the same language.",
        "folder": DATA_DIR / "axiom4_equivalent",
        "log": "equivalent_log.csv",
        "models": {
            "M_direct_choice": "model_direct_choice.pnml",
            "M_silent_routing": "model_silent_routing_choice.pnml",
        },
        "max_trace_length": 4,
        "checks": [{"type": "equality", "name": "A4_direct_eq_silent", "models": ["M_direct_choice", "M_silent_routing"]}],
    },
    {
        "name": "axiom4_complex_silent_routing",
        "type": "model_comparison",
        "axiom": "A4",
        "description": "Complex A4 case: a direct-branch model and a compact silent-routing model have the same visible language with optional skipped activities.",
        "folder": DATA_DIR / "axiom4_complex_silent_routing",
        "log": "axiom4_complex_log.csv",
        "models": {
            "M_direct_complex_choices": "model_direct_complex_choices.pnml",
            "M_silent_optional_routing": "model_silent_optional_routing.pnml",
        },
        "max_trace_length": 7,
        "checks": [{"type": "equality", "name": "A4_complex_direct_eq_silent", "models": ["M_direct_complex_choices", "M_silent_optional_routing"]}],
    },
    {
        "name": "axiom5_log_extension",
        "type": "log_extension",
        "axiom": "A5",
        "description": "Fixed model and increasingly complete fitting logs.",
        "folder": DATA_DIR / "axiom5_log_extension",
        "logs": {
            "L1_one_variant": "log_L1.csv",
            "L2_two_variants": "log_L2.csv",
            "L3_three_variants": "log_L3.csv",
        },
        "model": "model_choice_bde.pnml",
        "model_name": "M_choice_bde",
        "max_trace_length": 4,
        "checks": [{"type": "log_order", "name": "A5_L1_le_L2_le_L3", "order": ["L1_one_variant", "L2_two_variants", "L3_three_variants"], "direction": "non_decreasing"}],
    },
    {
        "name": "stochastic_frequency",
        "type": "log_extension",
        "axiom": "stochastic",
        "description": "Same log support but different frequencies. LMP should be unchanged; stochastic metrics can change.",
        "folder": DATA_DIR / "stochastic_frequency",
        "logs": {
            "balanced_50_50": "log_balanced.csv",
            "skewed_95_5": "log_skewed.csv",
        },
        "model": "model_exact_two_variants.pnml",
        "model_name": "M_exact_two_variants",
        "max_trace_length": 4,
        "checks": [],
    },
    {
        "name": "stochastic_near_miss",
        "type": "model_comparison",
        "axiom": "stochastic",
        "description": "A model with exact, near-miss, and extra traces. Compare JS and EMD.",
        "folder": DATA_DIR / "stochastic_near_miss",
        "log": "near_miss_log.csv",
        "models": {
            "M_near_miss": "model_with_near_miss_and_extra.pnml",
        },
        "max_trace_length": 6,
        "checks": [],
    },

    {
        "name": "axiom4_stochastic_semantics",
        "type": "model_comparison",
        "description": (
            "Two models have the same visible language, but different internal branching structures. "
            "Language-derived stochastic metrics should be equal; simulation-derived stochastic metrics may differ."
        ),
        "folder": DATA_DIR / "axiom4_stochastic_semantics",
        "log": "axiom4_stochastic_log.csv",
        "models": {
            "M_direct_equal_branches": "model_direct_equal_branches.pnml",
            "M_tau_unequal_simulation": "model_tau_unequal_simulation_probabilities.pnml",
        },
        "max_trace_length": 8,
        "axiom": "A4-stochastic",
        "checks": [{"type": "equality", "name": "A4_stochastic_language_equivalence", "models": ["M_direct_equal_branches", "M_tau_unequal_simulation"]}],
        "simulation": {
            "n_simulations": 20000,
            "max_steps": 50,
            "seed": 42,
        },
    },
]

## 9a. Advanced Axiom 4 case: direct choices versus silent routing

The basic Axiom 4 test checks two small models with the same visible language. The new experiment `axiom4_complex_silent_routing` makes this test more demanding.

The event log contains a realistic pattern with optional and skipped activities:

- `request_info` may be executed or skipped;
- `manual_review` may be executed or skipped;
- after the decision, approved cases may execute `notify` or skip it;
- rejected cases close directly.

The two models are intentionally different in structure:

### Model 1: direct visible choices

`M_direct_complex_choices` enumerates the complete visible branches directly. It has no silent routing transitions. Each complete variant is represented as a visible branch from the initial marking to the final marking.

### Model 2: silent optional routing

`M_silent_optional_routing` represents the same behavior compactly using silent transitions:

\[
\tau_{skip\_request\_info},
\tau_{skip\_manual\_review},
\tau_{skip\_notify}
\]

These silent transitions model routing decisions that are not recorded in the event log. For example, the absence of `manual_review` is not a visible event; it is represented by an internal skip.

The expected relation is:

\[
\mathcal L(M_{direct}) = \mathcal L(M_{\tau})
\]

Therefore, Axiom 4 requires:

\[
precision(L,M_{direct}) = precision(L,M_{\tau})
\]

This is a stronger test than the basic case because the two models differ not only by a small routing detail, but by their entire representation strategy: explicit branch enumeration versus compact silent skipping.

The notebook verifies this in two ways:

1. by comparing the generated bounded languages of the two models;
2. by comparing the precision values returned by each metric.

## 10. Experiment execution functions

In [14]:
def add_metric_row(rows, *, experiment, axiom, log_name, model_name, metric, score, time_s, status, details):
    row = {
        "experiment": experiment,
        "axiom": axiom,
        "log": log_name,
        "model": model_name,
        "metric": metric,
        "score": score,
        "time_s": time_s,
        "status": status,
    }
    row.update(details)
    rows.append(row)


def run_model_comparison_experiment(exp: Dict[str, object], prior="power_law", lambda_=0.7, alpha=2.0):
    folder = Path(exp["folder"])
    log_path = folder / exp["log"]
    max_len = int(exp.get("max_trace_length", DEFAULT_MAX_TRACE_LENGTH))

    df_log, event_log = load_csv_log_for_pm4py(log_path)
    log_counts = extract_variant_counts(df_log)
    log_variants = set(log_counts)

    model_data = {}
    for model_name, model_file in exp["models"].items():
        net, im, fm = load_petri_net_from_pnml(folder / model_file)
        start = time.perf_counter()
        language = compute_bounded_language_with_playout(net, im, fm, max_trace_length=max_len)
        playout_time = time.perf_counter() - start
        model_data[model_name] = {"net": net, "im": im, "fm": fm, "language": language, "playout_time_s": playout_time}

    shared_alphabet = infer_alphabet(log_variants, *(md["language"] for md in model_data.values()))
    rows = []

    for model_name, mdict in model_data.items():
        net, im, fm = mdict["net"], mdict["im"], mdict["fm"]
        language = mdict["language"]
        base_details = {
            "max_trace_length": max_len,
            "alphabet_size": len(shared_alphabet),
            "num_cases": sum(log_counts.values()),
            "num_log_variants": len(log_variants),
            "num_model_variants": len(language),
            "num_fitting_variants": len(log_variants.intersection(language)),
            "num_extra_model_variants": len(language.difference(log_variants)),
            "num_missing_log_variants": len(log_variants.difference(language)),
            "playout_time_s": mdict["playout_time_s"],
        }

        # LMP geometric and power-law
        for p_name, lam, alp in [("geometric", lambda_, alpha), ("power_law", lambda_, alpha)]:
            start = time.perf_counter()
            lmp = compute_lmp(log_variants, language, alphabet=shared_alphabet, prior=p_name, lambda_=lam, alpha=alp)
            elapsed = time.perf_counter() - start
            metric_name = f"LMP_{p_name}"
            if p_name == "geometric": metric_name += f"_lambda_{lam}"
            if p_name == "power_law": metric_name += f"_alpha_{alp}"
            details = dict(base_details)
            details.update({"numerator_mass": lmp["numerator_mass"], "denominator_mass": lmp["denominator_mass"]})
            add_metric_row(rows, experiment=exp["name"], axiom=exp.get("axiom"), log_name=Path(exp["log"]).stem, model_name=model_name,
                           metric=metric_name, score=lmp["score"], time_s=elapsed, status="ok", details=details)

        # PM4Py precision/fitness metrics
        for row in pm4py_metric_wrappers(event_log, net, im, fm):
            add_metric_row(rows, experiment=exp["name"], axiom=exp.get("axiom"), log_name=Path(exp["log"]).stem, model_name=model_name,
                           metric=row["metric"], score=row["score"], time_s=row["time_s"], status=row["status"], details=base_details)

        # Language-derived stochastic metrics
        for row in stochastic_metric_rows(log_counts, language, shared_alphabet, prior=prior, lambda_=lambda_, alpha=alpha):
            add_metric_row(rows, experiment=exp["name"], axiom=exp.get("axiom"), log_name=Path(exp["log"]).stem, model_name=model_name,
                           metric=row["metric"], score=row["score"], time_s=row["time_s"], status=row["status"], details=base_details)

        # Simulation-derived stochastic metrics
        # These probabilities depend on the internal structure of the Petri net.
        if "simulation" in exp:
            sim_cfg = exp.get("simulation", {})
            n_simulations = int(sim_cfg.get("n_simulations", 10000))
            sim_max_steps = int(sim_cfg.get("max_steps", 100))
            sim_seed = int(sim_cfg.get("seed", 42))

            start = time.perf_counter()
            sim_distribution, sim_diag = simulate_model_distribution(
                net,
                im,
                fm,
                n_simulations=n_simulations,
                max_steps=sim_max_steps,
                seed=sim_seed,
            )
            sim_distribution_time = time.perf_counter() - start

            sim_details = dict(base_details)
            sim_details.update({
                "simulation_n": n_simulations,
                "simulation_distribution_time_s": sim_distribution_time,
                "simulation_completed_runs": sim_diag["completed_runs"],
                "simulation_deadlocked_runs": sim_diag["deadlocked_runs"],
                "simulation_timeout_runs": sim_diag["timeout_runs"],
                "simulation_completion_rate": sim_diag["completion_rate"],
                "simulation_num_variants": sim_diag["num_simulated_variants"],
            })

            p_log = log_distribution_from_counts(log_counts)

            start = time.perf_counter()
            js_score = js_similarity(p_log, sim_distribution)
            js_elapsed = time.perf_counter() - start
            add_metric_row(rows, experiment=exp["name"], axiom=exp.get("axiom"), log_name=Path(exp["log"]).stem, model_name=model_name,
                           metric="stochastic_js_simulation_uniform_enabled", score=js_score, time_s=js_elapsed, status="ok", details=sim_details)

            start = time.perf_counter()
            emd_score, emd_status = earth_mover_similarity(p_log, sim_distribution)
            emd_elapsed = time.perf_counter() - start
            add_metric_row(rows, experiment=exp["name"], axiom=exp.get("axiom"), log_name=Path(exp["log"]).stem, model_name=model_name,
                           metric="stochastic_emd_simulation_uniform_enabled", score=emd_score, time_s=emd_elapsed, status=emd_status, details=sim_details)

    details = {"log_variants": log_variants, "log_counts": log_counts, "model_languages": {k:v["language"] for k,v in model_data.items()}, "alphabet": shared_alphabet}
    return pd.DataFrame(rows), details


def run_log_extension_experiment(exp: Dict[str, object], prior="power_law", lambda_=0.7, alpha=2.0):
    folder = Path(exp["folder"])
    max_len = int(exp.get("max_trace_length", DEFAULT_MAX_TRACE_LENGTH))
    model_name = exp.get("model_name", "M")
    net, im, fm = load_petri_net_from_pnml(folder / exp["model"])
    start = time.perf_counter()
    language = compute_bounded_language_with_playout(net, im, fm, max_trace_length=max_len)
    playout_time = time.perf_counter() - start

    log_objects = {}
    all_log_variants = []
    for log_name, log_file in exp["logs"].items():
        df_log, event_log = load_csv_log_for_pm4py(folder / log_file)
        log_counts = extract_variant_counts(df_log)
        log_objects[log_name] = {"df": df_log, "event_log": event_log, "counts": log_counts, "variants": set(log_counts)}
        all_log_variants.append(set(log_counts))

    shared_alphabet = infer_alphabet(language, *all_log_variants)
    rows = []

    for log_name, ldict in log_objects.items():
        log_counts = ldict["counts"]
        log_variants = ldict["variants"]
        event_log = ldict["event_log"]
        base_details = {
            "max_trace_length": max_len,
            "alphabet_size": len(shared_alphabet),
            "num_cases": sum(log_counts.values()),
            "num_log_variants": len(log_variants),
            "num_model_variants": len(language),
            "num_fitting_variants": len(log_variants.intersection(language)),
            "num_extra_model_variants": len(language.difference(log_variants)),
            "num_missing_log_variants": len(log_variants.difference(language)),
            "playout_time_s": playout_time,
        }

        for p_name, lam, alp in [("geometric", lambda_, alpha), ("power_law", lambda_, alpha)]:
            start = time.perf_counter()
            lmp = compute_lmp(log_variants, language, alphabet=shared_alphabet, prior=p_name, lambda_=lam, alpha=alp)
            elapsed = time.perf_counter() - start
            metric_name = f"LMP_{p_name}"
            if p_name == "geometric": metric_name += f"_lambda_{lam}"
            if p_name == "power_law": metric_name += f"_alpha_{alp}"
            details = dict(base_details)
            details.update({"numerator_mass": lmp["numerator_mass"], "denominator_mass": lmp["denominator_mass"]})
            add_metric_row(rows, experiment=exp["name"], axiom=exp.get("axiom"), log_name=log_name, model_name=model_name,
                           metric=metric_name, score=lmp["score"], time_s=elapsed, status="ok", details=details)

        for row in pm4py_metric_wrappers(event_log, net, im, fm):
            add_metric_row(rows, experiment=exp["name"], axiom=exp.get("axiom"), log_name=log_name, model_name=model_name,
                           metric=row["metric"], score=row["score"], time_s=row["time_s"], status=row["status"], details=base_details)

        for row in stochastic_metric_rows(log_counts, language, shared_alphabet, prior=prior, lambda_=lambda_, alpha=alpha):
            add_metric_row(rows, experiment=exp["name"], axiom=exp.get("axiom"), log_name=log_name, model_name=model_name,
                           metric=row["metric"], score=row["score"], time_s=row["time_s"], status=row["status"], details=base_details)

    details = {"model_language": language, "logs": log_objects, "alphabet": shared_alphabet}
    return pd.DataFrame(rows), details


def run_experiment(exp: Dict[str, object], prior="power_law", lambda_=0.7, alpha=2.0):
    exp_type = exp.get("type", "model_comparison")
    if exp_type == "model_comparison":
        return run_model_comparison_experiment(exp, prior=prior, lambda_=lambda_, alpha=alpha)
    if exp_type == "log_extension":
        return run_log_extension_experiment(exp, prior=prior, lambda_=lambda_, alpha=alpha)
    raise ValueError(f"Unknown experiment type: {exp_type}")

## 11. Axiom diagnostics

In [15]:
import numbers


def is_numeric(x) -> bool:
    return isinstance(x, numbers.Real) and not isinstance(x, bool) and not pd.isna(x)


def diagnostic_status_for_values(metric: str, lval, rval, ok):
    if metric.startswith("pm4py_fitness_"):
        return "not_applicable_fitness_metric"
    if not (is_numeric(lval) and is_numeric(rval)):
        return "not_numeric_score"
    return "pass" if ok else "fail"


def diagnostic_order(
    result_df: pd.DataFrame,
    order: Sequence[str],
    by: str = "model",
    direction: str = "non_increasing",
) -> pd.DataFrame:
    """
    Check metric ordering.

    direction="non_increasing" checks left >= right.
    direction="non_decreasing" checks left <= right.

    A2/A3 use non_increasing because the first model should be at least as precise.
    A5 uses non_decreasing because adding fitting log behaviour should not reduce precision.
    """
    if direction not in {"non_increasing", "non_decreasing"}:
        raise ValueError("direction must be 'non_increasing' or 'non_decreasing'")

    rows = []
    metrics = sorted(result_df["metric"].unique())

    for metric in metrics:
        sub = result_df[result_df["metric"] == metric]

        for left, right in zip(order, order[1:]):
            lv = sub.loc[sub[by] == left, "score"].values
            rv = sub.loc[sub[by] == right, "score"].values

            if len(lv) == 0 or len(rv) == 0:
                continue

            lval, rval = lv[0], rv[0]

            if is_numeric(lval) and is_numeric(rval):
                if direction == "non_increasing":
                    ok = lval >= rval
                    expected = f"{left} >= {right}"
                else:
                    ok = lval <= rval
                    expected = f"{left} <= {right}"
            else:
                ok = None
                expected = f"{left} >= {right}" if direction == "non_increasing" else f"{left} <= {right}"

            rows.append({
                "metric": metric,
                "expected": expected,
                "left_value": lval,
                "right_value": rval,
                "direction": direction,
                "axiom_respected": ok,
                "diagnostic_status": diagnostic_status_for_values(metric, lval, rval, ok),
            })

    return pd.DataFrame(rows)


def diagnostic_equality(result_df: pd.DataFrame, models: Sequence[str], tol: float = 1e-9) -> pd.DataFrame:
    """Check equality of metric values for a pair of language-equivalent models."""
    left, right = models
    rows = []
    metrics = sorted(result_df["metric"].unique())

    for metric in metrics:
        sub = result_df[result_df["metric"] == metric]
        lv = sub.loc[sub["model"] == left, "score"].values
        rv = sub.loc[sub["model"] == right, "score"].values

        if len(lv) == 0 or len(rv) == 0:
            continue

        lval, rval = lv[0], rv[0]

        if is_numeric(lval) and is_numeric(rval):
            ok = abs(lval - rval) <= tol
        else:
            ok = None

        rows.append({
            "metric": metric,
            "expected": f"{left} == {right}",
            "left_value": lval,
            "right_value": rval,
            "direction": "equality",
            "axiom_respected": ok,
            "diagnostic_status": diagnostic_status_for_values(metric, lval, rval, ok),
        })

    return pd.DataFrame(rows)


def run_diagnostics(exp: Dict[str, object], result_df: pd.DataFrame) -> pd.DataFrame:
    frames = []

    for check in exp.get("checks", []):
        if check["type"] == "order":
            d = diagnostic_order(
                result_df,
                check["order"],
                by="model",
                direction=check.get("direction", "non_increasing"),
            )
            d.insert(0, "check", check["name"])
            frames.append(d)

        elif check["type"] == "log_order":
            # A5: L1 <= L2 <= L3 in precision when fitting behaviour is added.
            d = diagnostic_order(
                result_df,
                check["order"],
                by="log",
                direction=check.get("direction", "non_decreasing"),
            )
            d.insert(0, "check", check["name"])
            frames.append(d)

        elif check["type"] == "equality":
            d = diagnostic_equality(result_df, check["models"])
            d.insert(0, "check", check["name"])
            frames.append(d)

    if frames:
        out = pd.concat(frames, ignore_index=True)
        out.insert(0, "experiment", exp["name"])
        return out

    return pd.DataFrame()

## 12. Run all controlled experiments

In [16]:
all_results = []
all_diagnostics = []
all_details = {}

for exp in EXPERIMENTS:
    print("=" * 100)
    print(exp["name"])
    print(exp.get("description", ""))
    print("=" * 100)

    result_df, details = run_experiment(exp, prior="power_law", lambda_=0.7, alpha=2.0)
    all_results.append(result_df)
    all_details[exp["name"]] = details

    display(result_df[["experiment", "log", "model", "metric", "score", "status", "time_s", "playout_time_s", "num_model_variants", "num_log_variants"]])

    diag = run_diagnostics(exp, result_df)
    if not diag.empty:
        all_diagnostics.append(diag)
        print("Diagnostics")
        display(diag)

results_df = pd.concat(all_results, ignore_index=True)
diagnostics_df = pd.concat(all_diagnostics, ignore_index=True) if all_diagnostics else pd.DataFrame()

precision_vs_fitness_three_models
M1 is smaller than the log, M2 equals the log language, M3 is larger.


replaying log with TBR, completed traces ::   0%|          | 0/3 [00:00<?, ?it/s]

computing precision with alignments, completed variants ::   0%|          | 0/3 [00:00<?, ?it/s]

/var/folders/l3/pr0xqvwx5xggq55v2bsw7vvm0000gn/T/ipykernel_2304/3234668650.py:4: DeprecatedWarning: precision_footprints is deprecated as of 2.3.0 and will be removed in 3.0.0. conformance checking using footprints will not be exposed in a future release
  value = func(*args, **kwargs)


replaying log with TBR, completed traces ::   0%|          | 0/2 [00:00<?, ?it/s]

...

replaying log with TBR, completed traces ::   0%|          | 0/4 [00:00<?, ?it/s]

aligning log, completed variants ::   0%|          | 0/4 [00:00<?, ?it/s]

,experiment,log,model,metric,score,status,time_s,playout_time_s,num_model_variants,num_log_variants
0,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,LMP_geometric_lambda_0.7,1.0,ok,0.000017,0.000726,4,4
1,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,LMP_power_law_alpha_2.0,1.0,ok,0.000059,0.000726,4,4
2,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,pm4py_precision_token_based_replay,1.0,ok,0.011262,0.000726,4,4
3,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,pm4py_precision_alignments,1.0,ok,0.021363,0.000726,4,4
4,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,pm4py_precision_footprints,1.0,ok,0.002685,0.000726,4,4
5,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,pm4py_fitness_token_based_replay,"{'perc_fit_traces': 25.0, 'average_trace_fitne...",ok,0.011125,0.000726,4,4
6,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,pm4py_fitness_alignments,"{'percFitTraces': 100.0, 'averageFitness': 1.0...",ok,0.067535,0.000726,4,4
7,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,stochastic_js_uniform,1.0,ok,0.000015,0.000726,4,4
8,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,stochastic_emd_uniform,1.0,ok,0.000627,0.000726,4,4
9,axiom4_stochastic_semantics,axiom4_stochastic_log,M_direct_equal_branches,stochastic_js_power_law_prior,0.716125,ok,0.000016,0.000726,4,4


Diagnostics


,experiment,check,metric,expected,left_value,right_value,direction,axiom_respected,diagnostic_status
0,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,LMP_geometric_lambda_0.7,M_direct_equal_branches == M_tau_unequal_simul...,1.0,1.0,equality,True,pass
1,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,LMP_power_law_alpha_2.0,M_direct_equal_branches == M_tau_unequal_simul...,1.0,1.0,equality,True,pass
2,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,pm4py_fitness_alignments,M_direct_equal_branches == M_tau_unequal_simul...,"{'percFitTraces': 100.0, 'averageFitness': 1.0...","{'percFitTraces': 100.0, 'averageFitness': 1.0...",equality,None,not_applicable_fitness_metric
3,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,pm4py_fitness_token_based_replay,M_direct_equal_branches == M_tau_unequal_simul...,"{'perc_fit_traces': 25.0, 'average_trace_fitne...","{'perc_fit_traces': 75.0, 'average_trace_fitne...",equality,None,not_applicable_fitness_metric
4,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,pm4py_precision_alignments,M_direct_equal_branches == M_tau_unequal_simul...,1.0,1.0,equality,True,pass
5,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,pm4py_precision_footprints,M_direct_equal_branches == M_tau_unequal_simul...,1.0,1.0,equality,True,pass
6,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,pm4py_precision_token_based_replay,M_direct_equal_branches == M_tau_unequal_simul...,1.0,1.0,equality,True,pass
7,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,stochastic_emd_power_law_prior,M_direct_equal_branches == M_tau_unequal_simul...,0.852365,0.852365,equality,True,pass
8,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,stochastic_emd_simulation_uniform_enabled,M_direct_equal_branches == M_tau_unequal_simul...,0.99917,0.968723,equality,False,fail
9,axiom4_stochastic_semantics,A4_stochastic_language_equivalence,stochastic_emd_uniform,M_direct_equal_branches == M_tau_unequal_simul...,1.0,1.0,equality,True,pass


## 13. Combined results and diagnostics

In [ ]:
results_df.head(20)

In [ ]:
diagnostics_df

## 13b. Axiom 3 flower: exact value versus bounded playout

The flower model is the only benchmark model whose language is infinite. Therefore, the bounded playout value is not the exact LMP value.

For a flower language over the chosen alphabet:

\[
\mathcal L(M_{\text{flower}})=\Sigma^*
\]

The denominator is known analytically. Under the geometric prior it is \(1\). Under the unnormalised power-law weights used in the notebook, it is \(\zeta(\alpha)\). The bounded playout denominator omits tail mass, so the bounded score overestimates the exact score.

In [ ]:
flower_exp = next(e for e in EXPERIMENTS if e["name"] == "axiom3_flower")
flower_details = all_details.get("axiom3_flower", {})

flower_log_variants = set(flower_details.get("log_variants", set()))
flower_alphabet = set(flower_details.get("alphabet", set()))

flower_exact_rows = []

if flower_log_variants and flower_alphabet:
    for prior_name, lam, alp, metric_name in [
        ("geometric", 0.7, 2.0, "LMP_geometric_lambda_0.7"),
        ("power_law", 0.7, 2.0, "LMP_power_law_alpha_2.0"),
    ]:
        exact = compute_lmp_against_full_sigma_star(
            flower_log_variants,
            flower_alphabet,
            prior=prior_name,
            lambda_=lam,
            alpha=alp,
        )

        bounded_rows = results_df[
            (results_df["experiment"] == "axiom3_flower")
            & (results_df["model"] == "M_flower_abcd")
            & (results_df["metric"] == metric_name)
        ]

        bounded_score = float(bounded_rows["score"].iloc[0]) if len(bounded_rows) else float("nan")
        bounded_denominator = float(bounded_rows["denominator_mass"].iloc[0]) if len(bounded_rows) and "denominator_mass" in bounded_rows else float("nan")

        flower_exact_rows.append({
            "experiment": "axiom3_flower",
            "model": "M_flower_abcd",
            "metric": metric_name,
            "bounded_K": flower_exp["max_trace_length"],
            "bounded_score": bounded_score,
            "exact_score": exact["score"],
            "relative_overestimate": (bounded_score / exact["score"] - 1.0) if exact["score"] > 0 else float("nan"),
            "bounded_denominator_mass": bounded_denominator,
            "exact_denominator_mass": exact["denominator_mass"],
            "numerator_mass": exact["numerator_mass"],
        })

flower_exact_df = pd.DataFrame(flower_exact_rows)
display(flower_exact_df)

## 13c. Automatically generated decoy summary table

The decoy table should be generated from `results_df`, not hard-coded in the text. This avoids inconsistencies between the research note and the CSV output.

In [ ]:
def make_decoy_summary_table(results_df: pd.DataFrame) -> pd.DataFrame:
    selected_metrics = [
        "LMP_geometric_lambda_0.7",
        "LMP_power_law_alpha_2.0",
        "pm4py_precision_token_based_replay",
        "pm4py_precision_alignments",
        "pm4py_precision_footprints",
    ]

    sub = results_df[
        (results_df["experiment"] == "axiom2_decoy_stress")
        & (results_df["metric"].isin(selected_metrics))
    ].copy()

    table = sub.pivot_table(
        index="metric",
        columns="model",
        values="score",
        aggfunc="first",
    ).reindex(selected_metrics)

    return table.reset_index()


decoy_summary_df = make_decoy_summary_table(results_df)
display(decoy_summary_df)

**What this shows.** The decoy stress test separates exact model-language precision from replay/structural diagnostics. LMP follows the language relation: the exact-language model receives a higher score even though it contains misleading dead visible branches. PM4Py precision metrics are useful diagnostics, but in this case they prefer the structurally cleaner model with a larger language.

**What this does not show.** It does not show that LMP replaces replay diagnostics. It shows that LMP measures a different target: the mass of complete model traces supported by the log.

## 13a. Language-equivalence verification for Axiom 4

For Axiom 4 experiments, the first thing to verify is not the metric value but the language relation itself:

\[
\mathcal L(M_1)=\mathcal L(M_2).
\]

The following cell compares the bounded languages generated by PM4Py playout for every experiment that declares an equality check. For the controlled A4 cases the symmetric difference should be empty.

In [ ]:
def language_equivalence_summary(all_details: Dict[str, object], experiments: Sequence[Dict[str, object]]) -> pd.DataFrame:
    rows = []
    for exp in experiments:
        for check in exp.get("checks", []):
            if check.get("type") != "equality":
                continue
            left, right = check["models"]
            details = all_details.get(exp["name"], {})
            languages = details.get("model_languages", {})
            if left not in languages or right not in languages:
                continue
            left_lang = languages[left]
            right_lang = languages[right]
            symdiff = left_lang.symmetric_difference(right_lang)
            rows.append({
                "experiment": exp["name"],
                "check": check["name"],
                "left_model": left,
                "right_model": right,
                "left_language_size": len(left_lang),
                "right_language_size": len(right_lang),
                "languages_equal": len(symdiff) == 0,
                "symmetric_difference_size": len(symdiff),
                "examples_in_symmetric_difference": sorted(symdiff)[:5],
            })
    return pd.DataFrame(rows)

language_equivalence_df = language_equivalence_summary(all_details, EXPERIMENTS)
display(language_equivalence_df)

**What this shows.** Axiom 4 is about visible model-language equivalence. Before interpreting metric values, the notebook verifies that the compared models generate the same bounded visible language. LMP must be invariant under this relation; simulation-derived stochastic metrics need not be invariant when the internal structure induces different trace probabilities.

## 14. Parameter sensitivity: geometric versus power-law LMP

This section recomputes only LMP over a grid of parameters.

In [ ]:
def run_lmp_sensitivity(exp: Dict[str, object], lambdas=(0.3,0.5,0.7,0.85,0.95), alphas=(1.2,1.5,2.0,3.0,5.0)):
    rows = []
    # Reuse experiment runner but filter LMP rows for each parameter.
    for lam in lambdas:
        df, _ = run_experiment(exp, prior="geometric", lambda_=lam, alpha=2.0)
        rows.append(df[df["metric"].str.startswith("LMP_geometric")])
    for alpha in alphas:
        df, _ = run_experiment(exp, prior="power_law", lambda_=0.7, alpha=alpha)
        rows.append(df[df["metric"].str.startswith("LMP_power_law")])
    return pd.concat(rows, ignore_index=True)

# Run sensitivity on selected examples only, to keep execution time reasonable.
sensitivity_experiments = [
    EXPERIMENTS[1],  # clean A2
    EXPERIMENTS[2],  # decoy stress
    EXPERIMENTS[5],  # A5 log extension
]

sensitivity_results = []
for exp in sensitivity_experiments:
    print("Running sensitivity:", exp["name"])
    sensitivity_results.append(run_lmp_sensitivity(exp))

sensitivity_df = pd.concat(sensitivity_results, ignore_index=True)
sensitivity_df[["experiment", "log", "model", "metric", "score", "num_model_variants", "num_log_variants"]]

**What this shows.** In language-inclusion cases, changing the prior changes numerical values but not the axiom-required ordering. For non-inclusion-related languages, the prior may legitimately change the ranking because it weights different regions of trace space differently. This is why the prior is declared and sensitivity-tested rather than hidden.

## 14b. Prior sensitivity outside language inclusion

The parameter sensitivity observed in the benchmark should be interpreted only under the relevant axiom premises. In the A2 cases, the model languages are nested, and the ordering is therefore stable for any fixed positive measure.

Outside language inclusion, there is no monotonicity axiom that fixes the ranking. The prior can legitimately participate in the decision. The following controlled example uses two non-inclusion-related finite languages: one model has a short extra trace, while the other has many longer extra traces. Changing \(\lambda\) changes the relative importance of long traces and can reverse the ranking.

In [ ]:
from itertools import product


def binary_traces_of_length(length: int, limit: int, alphabet=("a", "b")) -> Set[Trace]:
    traces = []
    for seq in product(alphabet, repeat=length):
        traces.append(tuple(seq))
        if len(traces) >= limit:
            break
    return set(traces)


non_inclusion_alphabet = {"a", "b"}
non_inclusion_log = {("a", "b")}

# M1: observed trace plus one short extra trace.
M1_non_inclusion = set(non_inclusion_log) | {("b",)}

# M2: observed trace plus many long extra traces.
M2_non_inclusion = set(non_inclusion_log) | binary_traces_of_length(7, 100, alphabet=("a", "b"))

prior_counterexample_rows = []

for lam in [0.70, 0.95]:
    for model_name, model_language in [
        ("M1_one_short_extra", M1_non_inclusion),
        ("M2_many_long_extras", M2_non_inclusion),
    ]:
        lmp = compute_lmp(
            non_inclusion_log,
            model_language,
            alphabet=non_inclusion_alphabet,
            prior="geometric",
            lambda_=lam,
            alpha=2.0,
        )
        prior_counterexample_rows.append({
            "lambda": lam,
            "model": model_name,
            "score": lmp["score"],
            "denominator_mass": lmp["denominator_mass"],
            "num_model_traces": len(model_language),
            "num_extra_traces": len(model_language - non_inclusion_log),
            "M1_subset_M2": M1_non_inclusion.issubset(M2_non_inclusion),
            "M2_subset_M1": M2_non_inclusion.issubset(M1_non_inclusion),
        })

prior_counterexample_df = pd.DataFrame(prior_counterexample_rows)
display(prior_counterexample_df.pivot(index="model", columns="lambda", values="score"))

print("M1 subset M2:", M1_non_inclusion.issubset(M2_non_inclusion))
print("M2 subset M1:", M2_non_inclusion.issubset(M1_non_inclusion))

## 15. Scalability probe

This section varies the playout bound \(K\) for selected experiments and records language size and runtime.

In [ ]:
def run_scalability_over_k(exp: Dict[str, object], k_values=(3,5,8,10,12)):
    rows = []
    for k in k_values:
        exp_copy = dict(exp)
        exp_copy["max_trace_length"] = k
        start = time.perf_counter()
        result_df, _ = run_experiment(exp_copy, prior="power_law", lambda_=0.7, alpha=2.0)
        total_time = time.perf_counter() - start
        summary = result_df.groupby(["experiment", "log", "model"], as_index=False).agg(
            model_language_size=("num_model_variants", "max"),
            playout_time_s=("playout_time_s", "max"),
        )
        summary["K"] = k
        summary["total_experiment_time_s"] = total_time
        rows.append(summary)
    return pd.concat(rows, ignore_index=True)

scalability_df = run_scalability_over_k(EXPERIMENTS[3], k_values=(1,2,3,4))  # flower example can grow quickly
scalability_df

**What this shows.** Bounded playout pays for traces. For cyclic or flower-like models, the generated bounded language grows quickly, and the bounded denominator can omit substantial tail mass. LMP is defined on the full language; bounded playout is an approximation unless the bound covers the full language.

## 15b. Methodological clarifications

The experiments should be interpreted with three methodological clarifications.

1. **Exact LMP versus bounded playout.** The mathematical LMP definition is over the full model language. PM4Py playout provides a bounded approximation. For finite benchmark languages this is often exact once \(K\) is large enough. For infinite languages, such as the flower model, bounded playout overestimates precision because the denominator omits unobserved tail mass.

2. **The prior is part of the metric.** The geometric or power-law prior is not a technical detail; it defines the reference measure used by LMP. In language-inclusion cases, A2 fixes the ordering for any fixed positive prior. For non-inclusion-related models, the ranking may depend on the prior and should therefore be fixed and declared in the experimental method.

3. **Fitness is not a precision diagnostic.** PM4Py fitness functions may return dictionaries rather than scalar precision values. The diagnostic table now marks these rows as not applicable for precision axioms instead of silently treating them as failed or skipping them.

## 16. Export results

In [ ]:
results_path = RESULTS_DIR / "metric_comparison_results.csv"
diagnostics_path = RESULTS_DIR / "axiom_diagnostics.csv"
sensitivity_path = RESULTS_DIR / "lmp_parameter_sensitivity.csv"
scalability_path = RESULTS_DIR / "scalability_probe.csv"
flower_exact_path = RESULTS_DIR / "axiom3_flower_exact_lmp.csv"
decoy_summary_path = RESULTS_DIR / "axiom2_decoy_summary.csv"
prior_counterexample_path = RESULTS_DIR / "prior_non_inclusion_counterexample.csv"

results_df.to_csv(results_path, index=False)
diagnostics_df.to_csv(diagnostics_path, index=False)
sensitivity_df.to_csv(sensitivity_path, index=False)
scalability_df.to_csv(scalability_path, index=False)

if "flower_exact_df" in globals() and not flower_exact_df.empty:
    flower_exact_df.to_csv(flower_exact_path, index=False)

if "decoy_summary_df" in globals() and not decoy_summary_df.empty:
    decoy_summary_df.to_csv(decoy_summary_path, index=False)

if "prior_counterexample_df" in globals() and not prior_counterexample_df.empty:
    prior_counterexample_df.to_csv(prior_counterexample_path, index=False)

print("Exported:")
print(" -", results_path)
print(" -", diagnostics_path)
print(" -", sensitivity_path)
print(" -", scalability_path)
if "flower_exact_df" in globals() and not flower_exact_df.empty:
    print(" -", flower_exact_path)
if "decoy_summary_df" in globals() and not decoy_summary_df.empty:
    print(" -", decoy_summary_path)
if "prior_counterexample_df" in globals() and not prior_counterexample_df.empty:
    print(" -", prior_counterexample_path)

## 17. Optional public-log workflow

Controlled logs are useful for testing axioms because the intended language relations are known.

For public logs, use this section as a template. Public-log experiments are useful for runtime and practical comparison, but they usually do not provide known model-language inclusion relations.

In [ ]:
# Example template; keep commented until a real log path is provided.

# PUBLIC_LOG_PATH = Path("path/to/public_log.xes")
# public_log = pm4py.read_xes(str(PUBLIC_LOG_PATH))
# process_tree = pm4py.discover_process_tree_inductive(public_log)
# net, im, fm = pm4py.convert_to_petri_net(process_tree)
#
# generated_language = compute_bounded_language_with_playout(net, im, fm, max_trace_length=10)
# print("Generated language size:", len(generated_language))

## Dedicated check: language equivalence versus simulation-derived stochastic equivalence

This cell focuses on the new `axiom4_stochastic_semantics` experiment.

The two models are designed to have the same visible language:

\[
\mathcal L(M_{direct})=\mathcal L(M_{tau})
\]

However, their internal routing structures are different. Under uniform random simulation over enabled transitions, the direct model gives approximately equal probability to the four complete branches, while the silent-routing model gives different probabilities because some traces require two local choices and others require only one.

Therefore:

\[
\mathcal L(M_{direct})=\mathcal L(M_{tau})
\]

but generally:

\[
\widehat P_{M_{direct}} \neq \widehat P_{M_{tau}}
\]

This is the key distinction: LMP is invariant under language equivalence, while simulation-based stochastic metrics are invariant only under equality of trace probability distributions.

In [ ]:
exp = next(e for e in EXPERIMENTS if e["name"] == "axiom4_stochastic_semantics")

result_df, details = run_experiment(
    exp,
    prior="power_law",
    lambda_=0.7,
    alpha=2.0,
)

display(result_df)

languages = details["model_languages"]
model_names = list(languages.keys())
language_equal = languages[model_names[0]] == languages[model_names[1]]

print()
print("Visible language equivalence:")
print(f"{model_names[0]} == {model_names[1]}:", language_equal)
print(f"Language size {model_names[0]}:", len(languages[model_names[0]]))
print(f"Language size {model_names[1]}:", len(languages[model_names[1]]))

folder = Path(exp["folder"])
df_log, event_log = load_csv_log_for_pm4py(folder / exp["log"])
log_distribution = empirical_log_distribution(df_log)

simulation_rows = []

for model_name, model_file in exp["models"].items():
    net, im, fm = load_petri_net_from_pnml(folder / model_file)
    sim_dist, sim_diag = simulate_model_distribution(
        net,
        im,
        fm,
        n_simulations=exp["simulation"]["n_simulations"],
        max_steps=exp["simulation"]["max_steps"],
        seed=exp["simulation"]["seed"],
    )

    for trace in sorted(set(log_distribution).union(sim_dist)):
        simulation_rows.append({
            "model": model_name,
            "trace": " -> ".join(trace),
            "log_probability": log_distribution.get(trace, 0.0),
            "simulation_probability": sim_dist.get(trace, 0.0),
        })

    print()
    print(model_name)
    print(sim_diag)

simulation_distribution_df = pd.DataFrame(simulation_rows)
display(simulation_distribution_df)

cols = [
    "model",
    "model_language_size",
    "LMP_power_law_alpha_2.0",
    "LMP_power_law_alpha_2",
    "js_uniform",
    "emd_uniform",
    "js_power_law_prior",
    "emd_power_law_prior",
    "js_simulation_uniform_enabled",
    "emd_simulation_uniform_enabled",
    "simulation_completion_rate",
]
cols = [c for c in cols if c in result_df.columns]
display(result_df[cols])

This will create an exports/axiom4_stochastic_semantics/ folder with the CSV files needed for the paper tables.

In [ ]:
from pathlib import Path
import pandas as pd


def export_axiom4_stochastic_semantics_results(
    results_dir: Path = Path("./results"),
    n_simulations: int = 20000,
    max_steps: int = 50,
    seed: int = 42,
):
    """
    Export the results of the Axiom 4 stochastic-semantics experiment.

    Files are saved in ./results, consistently with the other notebook exports.
    """

    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    exp = next(e for e in EXPERIMENTS if e["name"] == "axiom4_stochastic_semantics")

    result_df, details = run_experiment(
        exp,
        prior="power_law",
        lambda_=0.7,
        alpha=2.0,
    )

    # 1. Metric-level results
    metric_results_path = results_dir / "axiom4_stochastic_metric_results.csv"
    result_df.to_csv(metric_results_path, index=False)

    # 2. Language-equivalence diagnostics
    languages = details["model_languages"]
    model_names = list(languages.keys())

    if len(model_names) != 2:
        raise ValueError("This export function expects exactly two models.")

    m1, m2 = model_names
    lang1 = languages[m1]
    lang2 = languages[m2]

    language_rows = []

    for trace in sorted(lang1.union(lang2)):
        language_rows.append({
            "trace": " -> ".join(trace),
            f"in_{m1}": trace in lang1,
            f"in_{m2}": trace in lang2,
            "language_equivalent": lang1 == lang2,
        })

    language_df = pd.DataFrame(language_rows)

    language_path = results_dir / "axiom4_stochastic_language_equivalence.csv"
    language_df.to_csv(language_path, index=False)

    # 3. Empirical log distribution
    folder = Path(exp["folder"])
    df_log, event_log = load_csv_log_for_pm4py(folder / exp["log"])
    log_distribution = empirical_log_distribution(df_log)

    # 4. Simulation-derived distributions and diagnostics
    simulation_rows = []
    diagnostic_rows = []

    for model_name, model_file in exp["models"].items():
        net, im, fm = load_petri_net_from_pnml(folder / model_file)

        sim_dist, sim_diag = simulate_model_distribution(
            net,
            im,
            fm,
            n_simulations=n_simulations,
            max_steps=max_steps,
            seed=seed,
        )

        js_sim = js_similarity_from_distributions(
            log_distribution,
            sim_dist,
        )

        emd_sim = emd_similarity_from_distributions(
            log_distribution,
            sim_dist,
        )

        diagnostic_rows.append({
            "model": model_name,
            "js_simulation_uniform_enabled": js_sim,
            "emd_simulation_uniform_enabled": emd_sim,
            **sim_diag,
        })

        for trace in sorted(set(log_distribution).union(sim_dist)):
            simulation_rows.append({
                "model": model_name,
                "trace": " -> ".join(trace),
                "log_probability": log_distribution.get(trace, 0.0),
                "simulation_probability": sim_dist.get(trace, 0.0),
                "absolute_difference": abs(
                    log_distribution.get(trace, 0.0)
                    - sim_dist.get(trace, 0.0)
                ),
            })

    simulation_df = pd.DataFrame(simulation_rows)
    diagnostics_df = pd.DataFrame(diagnostic_rows)

    simulation_path = results_dir / "axiom4_stochastic_simulation_trace_probabilities.csv"
    diagnostics_path = results_dir / "axiom4_stochastic_simulation_diagnostics.csv"

    simulation_df.to_csv(simulation_path, index=False)
    diagnostics_df.to_csv(diagnostics_path, index=False)

    # 5. Compact summary
    selected_cols = [
        "experiment",
        "model",
        "model_language_size",
        "log_variants",
        "fitting_log_variants",
        "extra_model_variants",
        "LMP_geometric_lambda_0.7",
        "LMP_power_law_alpha_2.0",
        "stochastic_js_uniform",
        "stochastic_emd_uniform",
        "stochastic_js_power_law_prior",
        "stochastic_emd_power_law_prior",
        "js_simulation_uniform_enabled",
        "emd_simulation_uniform_enabled",
    ]

    selected_cols = [c for c in selected_cols if c in result_df.columns]

    summary_df = result_df[selected_cols].copy()
    summary_df["language_equivalent_to_other_model"] = lang1 == lang2

    if "model" in summary_df.columns:
        summary_df = summary_df.merge(
            diagnostics_df[
                [
                    "model",
                    "completion_rate",
                    "deadlock_rate",
                    "timeout_rate",
                    "num_simulated_variants",
                ]
            ],
            on="model",
            how="left",
        )

    summary_path = results_dir / "axiom4_stochastic_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print("Export completed.")
    print(f"Metric results:          {metric_results_path}")
    print(f"Language equivalence:    {language_path}")
    print(f"Trace probabilities:     {simulation_path}")
    print(f"Simulation diagnostics:  {diagnostics_path}")
    print(f"Summary:                 {summary_path}")

    return {
        "metric_results": result_df,
        "language_equivalence": language_df,
        "trace_probabilities": simulation_df,
        "simulation_diagnostics": diagnostics_df,
        "summary": summary_df,
    }


axiom4_stochastic_exports = export_axiom4_stochastic_semantics_results()